<a href="https://colab.research.google.com/github/Dende99-math/Visual-Place-Recognition-Project/blob/mixvpr-reranking/MixVPR_5_2_Retrieval_Reranking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 1. CHECK SHARED DRIVE DATA
# ============================================================

from google.colab import drive
import os

drive.mount("/content/drive")

DRIVE_LOG_DIR = "/content/drive/MyDrive/VPR_mixvpr_logs"

print("Folder exists:", os.path.isdir(DRIVE_LOG_DIR))

if not os.path.isdir(DRIVE_LOG_DIR):
    raise FileNotFoundError(
        "VPR_mixvpr_logs non visibile nel MyDrive del nuovo account."
    )

print("\nContenuto VPR_mixvpr_logs:")
for x in sorted(os.listdir(DRIVE_LOG_DIR)):
    print(" -", x)

Mounted at /content/drive
Folder exists: True

Contenuto VPR_mixvpr_logs:
 - 2026-08-21_16-05-09
 - 2026-08-21_16-13-08
 - 2026-08-21_16-16-12
 - 2026-08-21_16-22-00
 - 2026-08-22_08-24-28
 - 2026-08-22_08-32-54
 - 2026-08-22_08-36-47
 - 2026-08-22_08-40-05
 - 2026-08-22_08-41-57
 - 2026-08-22_08-48-09
 - 2026-08-22_08-51-06
 - 2026-08-22_08-57-28
 - dataset_run_mapping.json
 - results
 - retrieval_run_mapping.json


In [2]:
# ============================================================
# 2. SETUP PROJECT ON NEW RUNTIME
# ============================================================

import os
import json
import glob

REPO = "/content/Visual-Place-Recognition-Project"
BRANCH = "mixvpr-reranking"

DRIVE_LOG_DIR = "/content/drive/MyDrive/VPR_mixvpr_logs"

# Clone only if the repository is not already present
if not os.path.exists(REPO):

    !git clone --recursive \
        --branch {BRANCH} \
        https://github.com/Dende99-math/Visual-Place-Recognition-Project.git \
        {REPO}

%cd {REPO}

!git checkout {BRANCH}
!git pull origin {BRANCH}
!git submodule update --init --recursive

print("\nRepository ready.")
print("Branch:", BRANCH)
print("Drive logs:", DRIVE_LOG_DIR)

Cloning into '/content/Visual-Place-Recognition-Project'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 140 (delta 19), reused 15 (delta 15), pack-reused 97 (from 2)
Receiving objects: 100% (140/140), 1.35 MiB | 3.79 MiB/s, done.
Resolving deltas: 100% (24/24), done.
Submodule 'image-matching-models' (https://github.com/alexstoken/image-matching-models.git) registered for path 'image-matching-models'
Cloning into '/content/Visual-Place-Recognition-Project/image-matching-models'...
remote: Enumerating objects: 2853, done.        
remote: Counting objects: 100% (1256/1256), done.        
remote: Compressing objects: 100% (380/380), done.        
remote: Total 2853 (delta 992), reused 892 (delta 874), pack-reused 1597 (from 1)        
Receiving objects: 100% (2853/2853), 84.39 MiB | 23.20 MiB/s, done.
Resolving deltas: 100% (2021/2021), done.
Submodule path 'image-matching-models': che

In [3]:
# ============================================================
# 3. INSTALL DEPENDENCIES
# ============================================================

%cd /content/Visual-Place-Recognition-Project

!pip install -q -r image-matching-models/requirements.txt
!pip install -q faiss-cpu
!pip install -q gdown
!pip install -q openpyxl

import torch

print("\nPyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "GPU non disponibile. "
        "Prima di Image Matching devi usare un runtime GPU."
    )

/content/Visual-Place-Recognition-Project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.5/745.5 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.3/225.3 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 95.5 MB/s eta 0:00:00

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [4]:
# ============================================================
# 4. DOWNLOAD ONLY REQUIRED DATASETS
# ============================================================

import os
import subprocess

REPO = "/content/Visual-Place-Recognition-Project"
BASE_DATA = f"{REPO}/data"

os.makedirs(BASE_DATA, exist_ok=True)

datasets = {
    "tokyo_xs": {
        "id": "15QB3VNKj93027UAQWv7pzFQO1JDCdZj2",
        "check": f"{BASE_DATA}/tokyo_xs/test/database",
    },
    "sf_xs": {
        "id": "1tQqEyt3go3vMh4fj_LZrRcahoTbzzH-y",
        "check": f"{BASE_DATA}/sf_xs/test/database",
    },
    "svox": {
        "id": "16iuk8voW65GaywNUQlWAbDt6HZzAJ_t9",
        "check": f"{BASE_DATA}/svox/images/test/gallery",
    },
}

for name, info in datasets.items():

    if os.path.isdir(info["check"]):
        print(f"[STATUS] {name}: already available.")
        continue

    print("\n" + "=" * 60)
    print(f"Downloading {name}")
    print("=" * 60)

    zip_path = f"{BASE_DATA}/{name}.zip"

    subprocess.run(
        [
            "gdown",
            f"https://drive.google.com/uc?id={info['id']}",
            "-O",
            zip_path,
        ],
        check=True,
    )

    print(f"Extracting {name}...")

    subprocess.run(
        [
            "unzip",
            "-q",
            zip_path,
            "-d",
            BASE_DATA,
        ],
        check=True,
    )

    os.remove(zip_path)

    if not os.path.isdir(info["check"]):
        raise RuntimeError(
            f"Dataset {name} extracted but expected path is missing:\n"
            f"{info['check']}"
        )

    print(f"[SUCCESS] {name} ready.")


print("\n" + "=" * 60)
print("FINAL DATASET CHECK")
print("=" * 60)

paths = [
    f"{BASE_DATA}/sf_xs/test/database",
    f"{BASE_DATA}/sf_xs/test/queries",
    f"{BASE_DATA}/tokyo_xs/test/database",
    f"{BASE_DATA}/tokyo_xs/test/queries",
    f"{BASE_DATA}/svox/images/test/gallery",
    f"{BASE_DATA}/svox/images/test/queries_night",
    f"{BASE_DATA}/svox/images/test/queries_sun",
]

for path in paths:
    print(
        "OK     " if os.path.isdir(path) else "MISSING",
        path
    )


Extracting tokyo_xs...
[SUCCESS] tokyo_xs ready.

Extracting sf_xs...
[SUCCESS] sf_xs ready.

Extracting svox...
[SUCCESS] svox ready.

FINAL DATASET CHECK
OK      /content/Visual-Place-Recognition-Project/data/sf_xs/test/database
OK      /content/Visual-Place-Recognition-Project/data/sf_xs/test/queries
OK      /content/Visual-Place-Recognition-Project/data/tokyo_xs/test/database
OK      /content/Visual-Place-Recognition-Project/data/tokyo_xs/test/queries
OK      /content/Visual-Place-Recognition-Project/data/svox/images/test/gallery
OK      /content/Visual-Place-Recognition-Project/data/svox/images/test/queries_night
OK      /content/Visual-Place-Recognition-Project/data/svox/images/test/queries_sun


In [5]:
# ============================================================
# 5. LOAD AND VERIFY RETRIEVAL MAPPING
# ============================================================

import os
import json
import glob

DRIVE_LOG_DIR = "/content/drive/MyDrive/VPR_mixvpr_logs"
MAPPING_FILE = f"{DRIVE_LOG_DIR}/retrieval_run_mapping.json"

with open(MAPPING_FILE, "r") as f:
    retrieval_mapping = json.load(f)

print("[SUCCESS] Retrieval mapping loaded:\n")
print(json.dumps(retrieval_mapping, indent=2))

print("\n" + "=" * 70)
print("VERIFY L2 RUNS")
print("=" * 70)

for dataset, runs in retrieval_mapping.items():

    l2_dir = runs.get("L2")

    if l2_dir is None:
        print(f"{dataset:15} | L2 mapping missing")
        continue

    run_exists = os.path.isdir(l2_dir)
    info_exists = os.path.isfile(f"{l2_dir}/info.log")
    preds_count = len(glob.glob(f"{l2_dir}/preds/*.txt"))

    print(
        f"{dataset:15} | "
        f"run={run_exists} | "
        f"log={info_exists} | "
        f"preds={preds_count}"
    )

[SUCCESS] Retrieval mapping loaded:

{
  "sf_xs": {
    "L2": "/content/drive/MyDrive/VPR_mixvpr_logs/2026-08-22_08-24-28",
    "dotproduct": "/content/drive/MyDrive/VPR_mixvpr_logs/2026-08-22_08-32-54"
  },
  "tokyo_xs": {
    "L2": "/content/drive/MyDrive/VPR_mixvpr_logs/2026-08-22_08-36-47",
    "dotproduct": "/content/drive/MyDrive/VPR_mixvpr_logs/2026-08-22_08-40-05"
  },
  "svox_night": {
    "L2": "/content/drive/MyDrive/VPR_mixvpr_logs/2026-08-22_08-41-57",
    "dotproduct": "/content/drive/MyDrive/VPR_mixvpr_logs/2026-08-22_08-48-09"
  },
  "svox_sun": {
    "L2": "/content/drive/MyDrive/VPR_mixvpr_logs/2026-08-22_08-51-06",
    "dotproduct": "/content/drive/MyDrive/VPR_mixvpr_logs/2026-08-22_08-57-28"
  }
}

VERIFY L2 RUNS
sf_xs           | run=True | log=True | preds=1000
tokyo_xs        | run=True | log=True | preds=315
svox_night      | run=True | log=True | preds=823
svox_sun        | run=True | log=True | preds=854


In [6]:
# ============================================================
# STAGE 2 — IMAGE MATCHING
# Completa tutti i dataset prima del reranking
# ============================================================

import os
import glob
import subprocess

matchers = [
    "superpoint-lg",
    "superglue",
    "loftr"
]

CHUNK_SIZE = 100

for name, runs in retrieval_mapping.items():

    preds_dir = f"{runs['L2']}/preds"

    num_queries = len(
        glob.glob(f"{preds_dir}/*.txt")
    )

    print("\n" + "=" * 70)
    print(f"DATASET: {name.upper()} — {num_queries} queries")
    print("=" * 70)

    for matcher in matchers:

        matcher_dir = f"{preds_dir}_{matcher}"

        completed = len(
            glob.glob(f"{matcher_dir}/*.torch")
        )

        # Matcher already complete
        if completed == num_queries:

            print(
                f"[STATUS] Skipping {matcher.upper()} on {name}: "
                f"{completed}/{num_queries} complete."
            )

            continue

        print(
            f"\n[RUN] {matcher.upper()} on {name}"
            f" — currently {completed}/{num_queries}"
        )

        # Run in chunks
        for start in range(0, num_queries, CHUNK_SIZE):

            end = min(
                start + CHUNK_SIZE,
                num_queries
            )

            # Check which files already exist inside this chunk
            missing = [
                q for q in range(start, end)
                if not os.path.isfile(
                    f"{matcher_dir}/{q:03d}.torch"
                )
            ]

            # Entire chunk already done
            if not missing:
                continue

            print(
                f"  queries {start} -> {end - 1}"
            )

            subprocess.run(
                [
                    "python",
                    "match_queries_preds.py",
                    "--preds-dir", preds_dir,
                    "--matcher", matcher,
                    "--device", "cuda",
                    "--im-size", "512",
                    "--num-preds", "20",
                    "--start-query", str(start),
                    "--num-queries", str(end - start),
                ],
                check=True
            )

        completed = len(
            glob.glob(f"{matcher_dir}/*.torch")
        )

        print(
            f"[DONE] {matcher.upper()} on {name}: "
            f"{completed}/{num_queries}"
        )


print("\n" + "=" * 70)
print("FINAL IMAGE MATCHING STATUS")
print("=" * 70)

for name, runs in retrieval_mapping.items():

    preds_dir = f"{runs['L2']}/preds"
    num_queries = len(glob.glob(f"{preds_dir}/*.txt"))

    print(f"\n{name.upper()}")

    for matcher in matchers:

        completed = len(
            glob.glob(
                f"{preds_dir}_{matcher}/*.torch"
            )
        )

        print(
            f"{matcher:15} | "
            f"{completed}/{num_queries}"
        )


DATASET: SF_XS — 1000 queries
[STATUS] Skipping SUPERPOINT-LG on sf_xs: 1000/1000 complete.
[STATUS] Skipping SUPERGLUE on sf_xs: 1000/1000 complete.
[STATUS] Skipping LOFTR on sf_xs: 1000/1000 complete.

DATASET: TOKYO_XS — 315 queries
[STATUS] Skipping SUPERPOINT-LG on tokyo_xs: 315/315 complete.
[STATUS] Skipping SUPERGLUE on tokyo_xs: 315/315 complete.
[STATUS] Skipping LOFTR on tokyo_xs: 315/315 complete.

DATASET: SVOX_NIGHT — 823 queries
[STATUS] Skipping SUPERPOINT-LG on svox_night: 823/823 complete.
[STATUS] Skipping SUPERGLUE on svox_night: 823/823 complete.
[STATUS] Skipping LOFTR on svox_night: 823/823 complete.

DATASET: SVOX_SUN — 854 queries
[STATUS] Skipping SUPERPOINT-LG on svox_sun: 854/854 complete.
[STATUS] Skipping SUPERGLUE on svox_sun: 854/854 complete.

[RUN] LOFTR on svox_sun — currently 176/854
  queries 100 -> 199
  queries 200 -> 299
  queries 300 -> 399
  queries 400 -> 499
  queries 500 -> 599
  queries 600 -> 699
  queries 700 -> 799
  queries 800 -> 853

In [7]:
# ============================================================
# STAGE 3 — FINAL RE-RANKING + SAVE RESULTS
# ============================================================

import os
import re
import subprocess
import pandas as pd

RESULTS_DIR = "/content/drive/MyDrive/VPR_mixvpr_logs/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

matchers = [
    "superpoint-lg",
    "superglue",
    "loftr"
]

results = []

for name, runs in retrieval_mapping.items():

    preds_dir = f"{runs['L2']}/preds"

    print("\n" + "=" * 70)
    print(f"FINAL RE-RANKING: {name.upper()}")
    print("=" * 70)

    for matcher in matchers:

        inliers_dir = f"{preds_dir}_{matcher}"

        print(f"\n{name.upper()} + {matcher.upper()}")

        cmd = [
            "python",
            "reranking.py",
            "--preds-dir", preds_dir,
            "--inliers-dir", inliers_dir,
            "--num-preds", "20",
            "--recall-values", "1", "5", "10", "20"
        ]

        run = subprocess.run(
            cmd,
            capture_output=True,
            text=True
        )

        print(run.stdout)

        if run.returncode != 0:
            print(run.stderr)
            raise RuntimeError(
                f"Reranking failed: {name} + {matcher}"
            )

        match = re.search(
            r"R@1:\s*([\d.]+).*?"
            r"R@5:\s*([\d.]+).*?"
            r"R@10:\s*([\d.]+).*?"
            r"R@20:\s*([\d.]+)",
            run.stdout,
            re.S
        )

        if match:

            r1, r5, r10, r20 = map(
                float,
                match.groups()
            )

            results.append({
                "dataset": name,
                "matcher": matcher,
                "R@1": r1,
                "R@5": r5,
                "R@10": r10,
                "R@20": r20
            })

# Save final table
df = pd.DataFrame(results)

csv_path = f"{RESULTS_DIR}/reranking_results.csv"
xlsx_path = f"{RESULTS_DIR}/reranking_results.xlsx"

df.to_csv(csv_path, index=False)
df.to_excel(xlsx_path, index=False)

print("\n" + "=" * 70)
print("FINAL RE-RANKING RESULTS")
print("=" * 70)

print(df.to_string(index=False))

print("\nSaved:")
print(csv_path)
print(xlsx_path)


FINAL RE-RANKING: SF_XS

SF_XS + SUPERPOINT-LG
R@1: 81.2, R@5: 83.3, R@10: 83.7, R@20: 83.9


SF_XS + SUPERGLUE
R@1: 79.6, R@5: 82.6, R@10: 83.6, R@20: 83.9


SF_XS + LOFTR
R@1: 80.1, R@5: 82.9, R@10: 83.7, R@20: 83.9


FINAL RE-RANKING: TOKYO_XS

TOKYO_XS + SUPERPOINT-LG
R@1: 88.6, R@5: 92.4, R@10: 92.7, R@20: 93.7


TOKYO_XS + SUPERGLUE
R@1: 87.0, R@5: 92.7, R@10: 93.3, R@20: 93.7


TOKYO_XS + LOFTR
R@1: 89.8, R@5: 92.4, R@10: 93.7, R@20: 93.7


FINAL RE-RANKING: SVOX_NIGHT

SVOX_NIGHT + SUPERPOINT-LG
R@1: 82.4, R@5: 86.4, R@10: 87.1, R@20: 88.0


SVOX_NIGHT + SUPERGLUE
R@1: 81.3, R@5: 86.1, R@10: 87.0, R@20: 88.0


SVOX_NIGHT + LOFTR
R@1: 82.5, R@5: 86.8, R@10: 87.6, R@20: 88.0


FINAL RE-RANKING: SVOX_SUN

SVOX_SUN + SUPERPOINT-LG
R@1: 91.8, R@5: 95.0, R@10: 95.4, R@20: 95.9


SVOX_SUN + SUPERGLUE
R@1: 90.4, R@5: 94.4, R@10: 95.3, R@20: 95.9


SVOX_SUN + LOFTR
R@1: 93.4, R@5: 95.0, R@10: 95.3, R@20: 95.9


FINAL RE-RANKING RESULTS
   dataset       matcher  R@1  R@5  R@10  R@20
   

In [3]:
from google.colab import drive
import os
import json

drive.mount("/content/drive")

DRIVE_LOG_DIR = "/content/drive/MyDrive/VPR_mixvpr_logs"
RESULTS_DIR = f"{DRIVE_LOG_DIR}/results"

print("DRIVE_LOG_DIR exists:", os.path.isdir(DRIVE_LOG_DIR))
print("RESULTS_DIR exists:", os.path.isdir(RESULTS_DIR))

timing_file = f"{RESULTS_DIR}/matching_times.json"

print("Timing file path:")
print(timing_file)

print("Timing file exists:", os.path.isfile(timing_file))

Mounted at /content/drive
DRIVE_LOG_DIR exists: True
RESULTS_DIR exists: True
Timing file path:
/content/drive/MyDrive/VPR_mixvpr_logs/results/matching_times.json
Timing file exists: True


In [4]:
import json

timing_file = "/content/drive/MyDrive/VPR_mixvpr_logs/results/matching_times.json"

with open(timing_file, "r") as f:
    matching_times = json.load(f)

print(json.dumps(matching_times, indent=2))

{
  "sf_xs|superpoint-lg": {
    "total_seconds": 3874.4397683143616,
    "num_queries": 1000
  },
  "sf_xs|superglue": {
    "total_seconds": 1490.5790133476257,
    "num_queries": 1000
  }
}


In [5]:
import pandas as pd

rows = []

for key, values in matching_times.items():
    dataset, matcher = key.split("|")

    total_seconds = values["total_seconds"]
    num_queries = values["num_queries"]

    avg_time = total_seconds / num_queries

    rows.append({
        "Dataset": dataset,
        "Matcher": matcher,
        "Total Time (s)": total_seconds,
        "Num Queries": num_queries,
        "Avg Query Time (s)": avg_time
    })

timing_df = pd.DataFrame(rows)

display(timing_df)

,Dataset,Matcher,Total Time (s),Num Queries,Avg Query Time (s)
0,sf_xs,superpoint-lg,3874.439768,1000,3.874440
1,sf_xs,superglue,1490.579013,1000,1.490579
